# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

## Paso 1: Cálculo de TF-IDF para el corpus de turismo

En este paso se trabaja con un corpus pequeño (`01_corpus_turismo_500.txt`) para comprender el funcionamiento del modelo TF-IDF.

Primero se cargan los documentos y se realiza un preprocesamiento básico:
- Conversión a minúsculas
- Eliminación de caracteres especiales
- Tokenización del texto

Posteriormente, se calcula:
- TF (frecuencia de término): mide cuántas veces aparece una palabra en un documento
- IDF (frecuencia inversa): penaliza términos muy comunes en el corpus

Finalmente, se construye la matriz TF-IDF donde cada documento se representa como un vector de pesos, permitiendo su comparación con otros documentos.

In [1]:
import sys
!{sys.executable} -m pip install pandas scikit-learn -q

In [3]:
import os
import re
import math
from collections import Counter
import pandas as pd

ruta_corpus = "01_corpus_turismo_500.txt"

with open(ruta_corpus, "r", encoding="utf-8", errors="ignore") as f:
    lineas = f.readlines()

docs_turismo = [linea.strip() for linea in lineas if linea.strip()]

print(f"Documentos cargados: {len(docs_turismo)}")
docs_turismo[:3]

Documentos cargados: 500


['Otavalo es conocido por su mercado indígena y su artesanía Perfecto para rafting.',
 'La Laguna Quilotoa destaca por su color turquesa',
 'Vilcabamba atrae visitantes interesados en longevidad y naturaleza Una experiencia inolvidable.']

### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

## Paso 2: Carga del corpus Gutenberg 1000

En este paso se carga un corpus más grande compuesto por aproximadamente 1000 libros obtenidos de Project Gutenberg.

Cada archivo de texto corresponde a un documento independiente dentro del corpus. Este conjunto de datos permite evaluar el comportamiento de TF-IDF en un escenario más realista y de mayor escala.

Se almacenan tanto los textos como los nombres de los documentos para posteriormente poder identificar los resultados de las consultas.

In [8]:
carpeta_libros = "corpus_limpio"

lista_archivos = sorted([
    f for f in os.listdir(carpeta_libros)
    if f.endswith(".txt")
])

docs_gutenberg = []
nombres_docs = []

for nombre_archivo in lista_archivos:
    ruta_archivo = os.path.join(carpeta_libros, nombre_archivo)

    with open(ruta_archivo, "r", encoding="utf-8", errors="ignore") as f:
        contenido = f.read()

    if contenido.strip():
        docs_gutenberg.append(contenido)
        nombres_docs.append(nombre_archivo)

print(f"Libros cargados: {len(docs_gutenberg)}")
nombres_docs[:10]

Libros cargados: 14


['pg100.txt',
 'pg1342.txt',
 'pg145.txt',
 'pg1513.txt',
 'pg16389.txt',
 'pg2641.txt',
 'pg2701.txt',
 'pg37106.txt',
 'pg42538.txt',
 'pg45368.txt']

### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

## Paso 3: Cálculo de TF-IDF para Gutenberg

Se aplica el mismo proceso de cálculo de TF-IDF al corpus Gutenberg.

Debido al tamaño del corpus, este paso es más costoso computacionalmente, pero permite obtener representaciones vectoriales más completas.

El resultado es una matriz donde:
- Cada fila representa un documento
- Cada columna representa un término del vocabulario
- Cada valor corresponde al peso TF-IDF del término en el documento

Esta representación es clave para realizar búsquedas eficientes en grandes colecciones de texto.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

matriz_tfidf = vec.fit_transform(docs_gutenberg)

df_tfidf = pd.DataFrame(
    matriz_tfidf.toarray(),
    columns=vec.get_feature_names_out()
)

print(f"Dimensiones de la matriz TF-IDF: {df_tfidf.shape}")
df_tfidf.head()

Dimensiones de la matriz TF-IDF: (14, 5000)


,000,10,100,101,102,105,106,108,109,11,...,yield,yoke,yonder,york,young,younger,youngest,youth,youthful,zeal
0,0.000049,0.000129,0.000223,0.00008,0.000087,0.000087,0.000087,0.000161,0.00008,0.000069,...,0.009709,0.002971,0.006263,0.037839,0.025933,0.001897,0.001871,0.016232,0.002067,0.002196
1,0.000331,0.000433,0.000000,0.00000,0.000000,0.000000,0.000000,0.000539,0.00000,0.000000,...,0.001618,0.000000,0.000000,0.000500,0.043996,0.009901,0.007439,0.003536,0.000000,0.000000
2,0.000130,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000638,0.002123,0.000000,0.000788,0.044590,0.002648,0.000916,0.003206,0.001708,0.000683
3,0.000777,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.001265,0.007593,0.000000,0.018649,0.001661,0.001092,0.004984,0.003054,0.000000
4,0.000393,0.001029,0.001186,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.001103,...,0.000000,0.000000,0.000000,0.000000,0.027086,0.002518,0.000000,0.002937,0.000000,0.001029


### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

## Paso 4: Función de búsqueda

Se implementa una función de búsqueda que permite ingresar una consulta en lenguaje natural.

El proceso consiste en:
1. Tokenizar la consulta
2. Buscar los términos dentro del vocabulario
3. Calcular un puntaje sumando los valores TF-IDF de cada término en cada documento
4. Ordenar los documentos según su relevancia

Esto permite recuperar los documentos más relevantes para una consulta dada, simulando el funcionamiento básico de un motor de búsqueda.

In [10]:
def tokenizar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto)
    return texto.split()


def buscar(consulta, top_n=10):
    tokens = tokenizar(consulta)
    puntajes = []

    for idx in range(len(docs_gutenberg)):
        total = 0

        for termino in tokens:
            if termino in df_tfidf.columns:
                total += df_tfidf.iloc[idx][termino]

        puntajes.append((nombres_docs[idx], total))

    puntajes.sort(key=lambda x: x[1], reverse=True)

    return pd.DataFrame(puntajes[:top_n], columns=["Documento", "Score TF-IDF"])

In [11]:
buscar("love marriage family", top_n=10)

,Documento,Score TF-IDF
0,pg100.txt,0.128282
1,pg1513.txt,0.127433
2,pg84.txt,0.119648
3,pg1342.txt,0.106185
4,pg145.txt,0.064799
5,pg37106.txt,0.058037
6,pg2641.txt,0.038613
7,pg42538.txt,0.032587
8,pg16389.txt,0.032189
9,pg45368.txt,0.029649


## Interpretación de resultados

Al realizar la consulta "love marriage family", se observa que los documentos con mayor puntaje TF-IDF contienen estos términos con mayor frecuencia y relevancia.

Los primeros resultados corresponden a documentos donde estos conceptos son más representativos, lo cual indica que el modelo TF-IDF está funcionando correctamente al priorizar términos importantes dentro del corpus.

Esto demuestra que TF-IDF permite recuperar documentos relevantes basándose en la importancia de las palabras dentro de cada documento.